In [2]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


In [3]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [4]:
X_train_tensor = torch.tensor(X_train, dtype=torch.bfloat16)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.bfloat16)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [5]:
batch_size = 16
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=True)

In [6]:
_,(X,y) = next(enumerate(train_loader))
X[:,:,0].shape, X.shape

(torch.Size([16, 36]), torch.Size([16, 36, 6]))

In [7]:
from uni2ts.eval_util.data import get_gluonts_test_dataset
from uni2ts.eval_util.plot import plot_next_multi
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/uni2ts/common/env.py:43: UserWarning: Failed to load .env file.
  warnings.warn("Failed to load .env file.")


In [9]:
model = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small")
model.to("cuda")
with torch.no_grad():
    outputs = model.encoder(X.to("cuda"))


RuntimeError: The size of tensor a (6) must match the size of tensor b (384) at non-singleton dimension 2

In [10]:

device = "cuda"

model.eval()


X = X.to(device)                  # (B,T,C)
mask = torch.ones(X.shape[:2], device=device)   # (B,T)

with torch.no_grad():
    outputs = model(
        X,
        observed_mask=mask
    )


TypeError: MoiraiModule.forward() missing 5 required positional arguments: 'sample_id', 'time_id', 'variate_id', 'prediction_mask', and 'patch_size'

In [23]:
import torch
import torch.nn as nn
from uni2ts.model.moirai import MoiraiModule

# ---- config façon notebook
SIZE = "small"       # small/base/large
BSZ = 32
NUM_CLASSES = 5
DROPOUT = 0.2

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- load backbone (Moirai 1.x)
backbone = MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}").to(device)
backbone.eval()

def masked_mean_pool(h, mask):
    # h: (B,T,H), mask: (B,T) {0,1}
    m = mask.unsqueeze(-1)               # (B,T,1)
    h = h * m
    denom = m.sum(dim=1).clamp(min=1.0)  # (B,1)
    return h.sum(dim=1) / denom          # (B,H)

class MoiraiClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.head = None
        self.num_classes = num_classes

    def forward(self, x, mask):
        """
        x: (B,T,C) multivarié
        mask: (B,T) 1=valide 0=pad
        """
        # Certaines versions acceptent observed_mask; sinon on fallback
        try:
            out = self.backbone(x, observed_mask=mask)
        except TypeError:
            out = self.backbone(x)

        # ---- récupérer un tenseur (B,T,H)
        if isinstance(out, dict):
            # selon versions, l'une de ces clés existe
            h = out.get("hidden_states") or out.get("encoder_hidden_states") or out.get("last_hidden_state")
        else:
            # parfois l'output est déjà un tenseur
            h = out

        if h is None:
            raise RuntimeError(f"Je n'ai pas trouvé les hidden states. Keys: {list(out.keys())}")

        if h.ndim == 2:
            emb = h
        else:
            emb = masked_mean_pool(h, mask)

        emb = self.dropout(emb)

        # init lazy head
        if self.head is None:
            H = emb.shape[-1]
            self.head = nn.Sequential(
                nn.LayerNorm(H),
                nn.Linear(H, self.num_classes)
            ).to(emb.device)

        return self.head(emb)

model = MoiraiClassifier(backbone, NUM_CLASSES, dropout=DROPOUT).to(device)


In [24]:
B, T, C = 4, 128, 6
X = torch.randn(B, T, C, device=device)
mask = torch.ones(B, T, device=device)

with torch.no_grad():
    try:
        out = backbone(X, observed_mask=mask)
    except TypeError:
        out = backbone(X)

print(type(out))
if isinstance(out, dict):
    print("keys:", out.keys())
else:
    print("tensor shape:", out.shape)


TypeError: MoiraiModule.forward() missing 6 required positional arguments: 'observed_mask', 'sample_id', 'time_id', 'variate_id', 'prediction_mask', and 'patch_size'